# 02 - Selection versus generation, on one encoder

The comparison that isolates the claim: one encoder, one training loop, one seed,
one dataset, and **only the output interface differs**. `span` emits two indices;
`generative` emits characters over a closed 70-character alphabet.

This notebook trains both at a reduced scale so it runs in a few minutes. The
shipped numbers come from `scripts/run_experiments.py` at full scale; the tables
read from `results/tables/` at the bottom are those.

In [1]:
import sys, os
os.environ.setdefault("OMP_NUM_THREADS", "2")
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
import torch; torch.set_num_threads(2)
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)
print("torch", torch.__version__)


torch 2.13.0+cpu


In [2]:
from gdx.config import load_config
from gdx.pipelines.core import prepare, train_head
from gdx.arms import ARM_BY_NAME, model_candidates
from gdx.pipelines.core import evaluate_arm

cfg = load_config("../configs/base.yaml", ["data.n_train=400", "data.n_val=100", "data.n_test=150", "optim.epochs=4"])
prepared = prepare(cfg, seed=0)
print(prepared.splits.sizes)

{'train': 400, 'val': 100, 'test': 150}


In [3]:
models, candidates = {}, {}
for head in ("span", "generative"):
    models[head], hist = train_head(cfg, prepared, head)
    candidates[head] = model_candidates(models[head], prepared.splits.test, cfg)
    print(head, "params", hist["n_params"], "train seconds", round(hist["train_seconds"], 1))

11:17:12 INFO    training head=span seed=0 params=108892


11:17:25 INFO    epoch 1/4 train 4.2479 val 3.2396 (13.4s)


11:17:51 INFO    epoch 2/4 train 2.6130 val 1.9182 (39.5s)


11:18:08 INFO    epoch 3/4 train 1.8102 val 1.5514 (56.3s)


11:18:27 INFO    epoch 4/4 train 1.5920 val 1.4567 (75.4s)


...

11:19:26 INFO    epoch 1/4 train 3.3585 val 2.5768 (56.7s)


11:20:26 INFO    epoch 2/4 train 2.2804 val 2.0228 (117.3s)


11:21:20 INFO    epoch 3/4 train 1.9002 val 1.7979 (171.2s)


11:22:15 INFO    epoch 4/4 train 1.7601 val 1.7294 (226.5s)


generative params 201142 train seconds 226.5


In [4]:
rows = []
for arm_name in ("generative", "generative_verify", "span_only", "span_verify", "span_verify_norm"):
    arm = ARM_BY_NAME[arm_name]
    result = evaluate_arm(arm, prepared.splits.test, candidates[arm.source], cfg)
    rows.append({
        "arm": arm_name,
        "strict": result.summary["strict_accuracy"],
        "canonical": result.summary["canonical_accuracy"],
        "coverage": result.summary["coverage"],
        "hallucination": result.summary["hallucination_rate"],
        "grounding_exact": result.summary["grounding_exact"],
    })
pd.DataFrame(rows).round(4)

,arm,strict,canonical,coverage,hallucination,grounding_exact
0,generative,0.0000,0.0000,1.0000,1.0,NaN
1,generative_verify,0.0917,0.0917,0.0000,NaN,NaN
2,span_only,0.4375,0.4517,0.9283,0.0,0.4880
3,span_verify,0.4667,0.4933,0.5742,0.0,0.7452
4,span_verify_norm,0.4933,0.4933,0.5742,0.0,0.7452


The `hallucination` column is the point. For the span arms it is exactly 0 and
cannot be otherwise. For `generative` it is whatever the decoder produced.

`generative_verify` is the arm a fair reading demands: the *same* verification
loop applied to generated strings. If it also reaches 0, then the check rather
than the head is what removes hallucinated values -- and that is reported rather
than avoided.

In [5]:
# What the generative head actually emits, next to the truth.
gen = models["generative"]
batch = prepared.splits.test.collate(list(range(6)))
preds = gen.predict(batch)
rows = []
for doc, per_doc in zip(batch.docs, preds):
    for pred in per_doc:
        truth = doc.fields[pred.field_name]
        rows.append({
            "doc": doc.doc_id, "field": pred.field_name,
            "generated": pred.text, "target": truth.value,
            "in_document": doc.contains_value(pred.field_name, pred.text) if pred.text else None,
        })
pd.DataFrame(rows).head(24)

,doc,field,generated,target,in_document
0,500,invoice_id,IN-0638,DOC-92781,False
1,500,invoice_date,2023-02-0,2024-05-31,False
2,500,due_date,2023-02-0,,False
3,500,vendor_name,arrarial,Silverpine Logistics,False
4,500,po_number,ORD-0232,ORD-74289,False
5,500,subtotal,$ 2324.0.9,,False
6,500,tax,$ 23.8.8.,"$1,805.45",False
7,500,total,$ 2324.0.9,"$19,859.94",False
8,501,invoice_id,IN-0638,IN-48686,False
9,501,invoice_date,2023-02-0,2023-08-25,False


### The shipped tables

Everything below is read from `results/tables/`, produced by
`scripts/run_experiments.py` and `scripts/analyse.py` at full scale over three
seeds.

In [6]:
import pathlib
T = pathlib.Path("../results/tables")
pd.read_csv(T / "method_comparison.csv")

,arm,family,strict_accuracy,canonical_accuracy,coverage,hallucination_rate,grounding_exact,grounding_iou,precision,recall,f1,anls,absent_abstain_rate,error_auroc,aurc,ece,n_records
0,heuristic,baseline,0.762500,0.826875,0.755000,0.000000,0.965232,0.967056,0.965232,0.808039,0.879668,0.751052,1.000000,0.670352,0.020874,0.140415,3200.0
1,llm_stub,baseline,0.756563,0.756563,0.745938,0.000000,NaN,NaN,0.882698,0.730076,0.799166,0.746474,1.000000,NaN,NaN,NaN,3200.0
2,generative,reference,0.049375,0.049375,0.890938,0.998597,NaN,NaN,0.000702,0.000693,0.000697,0.278196,0.496815,0.845911,0.998286,0.416965,3200.0
3,generative_verify,baseline,0.098750,0.098750,0.001250,0.000000,NaN,NaN,0.500000,0.000693,0.001384,0.001213,1.000000,0.750000,0.270833,0.066773,3200.0
4,span_only,ablation,0.875938,0.939375,0.916250,0.000000,0.962317,0.965109,0.940655,0.955648,0.948092,0.890199,0.789809,0.905418,0.008535,0.024006,3200.0
5,span_verify,ours,0.882500,0.948750,0.864062,0.000000,0.990929,0.991063,0.987703,0.946292,0.966555,0.876399,0.971338,0.905755,0.001377,0.069292,3200.0
6,span_verify_norm,ours,0.948125,0.948750,0.864062,0.000000,0.990929,0.991063,0.987703,0.946292,0.966555,0.950724,0.971338,0.905755,0.001377,0.069292,3200.0


In [7]:
pd.read_csv(T / "verdicts.csv")

,arm,reference,metric,value,reference_value,delta,noise_scale,ratio_to_noise,verdict
0,generative_verify,generative,strict_accuracy,0.092292,0.047708,0.044583,0.007959,5.601595,robust
1,heuristic,generative,strict_accuracy,0.759271,0.047708,0.711562,0.006515,109.215647,robust
2,llm_stub,generative,strict_accuracy,0.757396,0.047708,0.709688,0.006515,108.927859,robust
3,span_only,generative,strict_accuracy,0.881771,0.047708,0.834063,0.013530,61.643394,robust
4,span_verify,generative,strict_accuracy,0.888750,0.047708,0.841042,0.012725,66.096008,robust
5,span_verify_norm,generative,strict_accuracy,0.954063,0.047708,0.906354,0.013412,67.577652,robust
6,generative_verify,generative,canonical_accuracy,0.092292,0.047708,0.044583,0.007959,5.601595,robust
7,heuristic,generative,canonical_accuracy,0.824375,0.047708,0.776667,0.006515,119.208295,robust
8,llm_stub,generative,canonical_accuracy,0.757396,0.047708,0.709688,0.006515,108.927859,robust
9,span_only,generative,canonical_accuracy,0.944375,0.047708,0.896667,0.012247,73.212527,robust


In [8]:
pd.read_csv(T / "statistical_tests.csv")

,name_a,name_b,metric,unit,mean_a,mean_b,difference,ci_lower,ci_upper,p_value,p_adjusted,effect_size,n,significant
0,heuristic,generative,hallucination_rate,set,0.000000,0.998597,-0.998344,-0.999586,-0.996689,NaN,NaN,NaN,2000,True
1,llm_stub,generative,hallucination_rate,set,0.000000,0.998597,-0.998324,-0.999581,-0.996230,NaN,NaN,NaN,2000,True
2,generative_verify,generative,hallucination_rate,set,0.000000,0.998597,-1.000000,-1.000000,-1.000000,NaN,NaN,NaN,2000,True
3,span_only,generative,hallucination_rate,set,0.000000,0.998597,-0.998597,-0.999649,-0.997194,NaN,NaN,NaN,2000,True
4,span_verify,generative,hallucination_rate,set,0.000000,0.998597,-0.998553,-0.999638,-0.997107,NaN,NaN,NaN,2000,True
5,span_verify_norm,generative,hallucination_rate,set,0.000000,0.998597,-0.998553,-0.999638,-0.997107,NaN,NaN,NaN,2000,True
6,heuristic,generative,strict_accuracy,document,0.762500,0.049375,0.713125,0.700313,0.725625,6.529134e-69,7.834961e-68,5.329589,400,True
7,heuristic,generative,canonical_accuracy,document,0.826875,0.049375,0.777500,0.765000,0.789070,6.019471e-70,1.023310e-68,6.351578,400,True
8,heuristic,generative,coverage,document,0.755000,0.890938,-0.135937,-0.149062,-0.123125,4.694365e-51,3.286055e-50,-1.072894,400,True
9,llm_stub,generative,strict_accuracy,document,0.756563,0.049375,0.707187,0.695312,0.719688,1.851638e-68,1.666474e-67,5.411643,400,True
